In [1]:
import requests
import bs4

In [ ]:
import requests
import pandas as pd
import time
from pathlib import Path

URL = "https://orchestrator.pgatour.com/graphql"

HEADERS = {
    "accept": "application/graphql-response+json, application/json",
    "accept-language": "en-US,en;q=0.6",
    "content-type": "application/json",
    "x-api-key": "da2-gsrx5bibzbb4njvhl7t37wqyl4",
    "x-pgat-platform": "web"
}

QUERY = """
query StatDetails($tourCode: TourCode!, $statId: String!, $year: Int, $eventQuery: StatDetailEventQuery) {
  statDetails(tourCode: $tourCode, statId: $statId, year: $year, eventQuery: $eventQuery) {
    statTitle
    statHeaders
    tourAvg
    rows {
      ... on StatDetailsPlayer {
        __typename
        playerId
        playerName
        country
        rank
        stats {
          statName
          statValue
        }
      }
    }
  }
}
"""

# stat IDs to collect — add more as you find them in the network tab
STAT_IDS = {
    "120":   "scoring_average",
    "02675":   "sg_total",
    "101":   "driving_distance",
    "103": "greens_in_regulation_percentage",
    "02568": "sg_approach",
    "02564": "sg_putting",
    "103": "scrambling_percentage",
    "150": "birdie_average",
}

YEARS = range(2019, 2026)


def fetch_stat(stat_id: str, year: int) -> dict:
    payload = {
        "operationName": "StatDetails",
        "query": QUERY,
        "variables": {
            "tourCode": "R",
            "statId": stat_id,
            "year": year,
            "eventQuery": None
        }
    }
    response = requests.post(URL, json=payload, headers=HEADERS)
    response.raise_for_status()
    return response.json()


def parse_rows(data: dict, stat_label: str, year: int) -> list:
    records = []
    rows = data.get("data", {}).get("statDetails", {}).get("rows", [])
    for row in rows:
        if row.get("__typename") != "StatDetailsPlayer":
            continue
        record = {
            "year": year,
            "stat": stat_label,
            "player_id": row["playerId"],
            "player_name": row["playerName"],
            "country": row["country"],
            "rank": row["rank"],
        }
        for s in row["stats"]:
            record[s["statName"]] = s["statValue"]
        records.append(record)
    return records


def get_output_path() -> Path:
    # Notebook may run from /notebooks, so detect project root safely.
    cwd = Path.cwd()
    if (cwd / "data").exists():
        project_root = cwd
    elif (cwd.parent / "data").exists():
        project_root = cwd.parent
    else:
        project_root = cwd
    output_path = project_root / "data" / "raw" / "pga_stats.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    return output_path


def main():
    all_records = []

    for stat_id, stat_label in STAT_IDS.items():
        for year in YEARS:
            print(f"Fetching {stat_label} ({stat_id}) — {year}...")
            try:
                data = fetch_stat(stat_id, year)
                records = parse_rows(data, stat_label, year)
                all_records.extend(records)
                print(f"  Got {len(records)} rows")
            except Exception as e:
                print(f"  Failed: {e}")
            time.sleep(0.5)

    df = pd.DataFrame(all_records)
    output_path = get_output_path()
    df.to_csv(output_path, index=False)
    print(f"\nDone. Saved {len(df)} rows to {output_path}")


if __name__ == "__main__":
    main()

Fetching scoring_average (120) — 2019...
  Got 188 rows
Fetching scoring_average (120) — 2020...
  Got 193 rows
Fetching scoring_average (120) — 2021...
  Got 196 rows
Fetching scoring_average (120) — 2022...
  Got 191 rows
Fetching scoring_average (120) — 2023...
  Got 193 rows
Fetching scoring_average (120) — 2024...
  Got 184 rows
Fetching scoring_average (120) — 2025...
  Got 179 rows
Fetching sg_total (02675) — 2019...
  Got 188 rows
Fetching sg_total (02675) — 2020...
  Got 193 rows
Fetching sg_total (02675) — 2021...
  Got 196 rows
Fetching sg_total (02675) — 2022...
  Got 193 rows
Fetching sg_total (02675) — 2023...
  Got 193 rows
Fetching sg_total (02675) — 2024...
  Got 184 rows
Fetching sg_total (02675) — 2025...
  Got 180 rows
Fetching driving_distance (101) — 2019...
  Got 188 rows
Fetching driving_distance (101) — 2020...
  Got 193 rows
Fetching driving_distance (101) — 2021...
  Got 196 rows
Fetching driving_distance (101) — 2022...
  Got 193 rows
Fetching driving_distan

OSError: Cannot save file into a non-existent directory: 'data\raw'